# RAG Pipeline — Error-Based Retrieval for SFT Qwen 2.5 Coder 7B

This notebook uses **error/traceback-based RAG retrieval** to fix runtime failures from the fine-tuned Qwen model's predictions.

### Workflow

1. **Load smoke_report** — Read `smoke_report.json` from the fine-tuned eval runtime, filter `ok=false` entries with real tracebacks  
2. **Read failed prediction code** — Load `prediction.py` for each failed sample  
3. **Error-based RAG retrieval** — Use the **traceback/error text** (not the code) as the query to ChromaDB → rerank with cross-encoder  
4. **Prompt LLM** — System prompt (bug-fixer) + user prompt (failed code + traceback + relevant docs)  
5. **Parse & Save** — Extract corrected code, save all artifacts to `RAG_outputs/RAG_with_Baseline/`

### Components

| Component | Choice |
|-----------|--------|
| **LLM** | `qwen/qwen-2.5-coder-7b-instruct` via OpenRouter |
| **Embedding** | `BAAI/bge-base-en-v1.5` (768-dim) |
| **Reranker** | `BAAI/bge-reranker-base` |
| **Vector DB** | ChromaDB (persistent, cosine) |
| **RAG query source** | Runtime error / traceback text |

## 1 — Install & Import Dependencies

In [1]:
# Install required packages (uncomment on first run)
# !pip install chromadb sentence-transformers langchain-openai python-dotenv datasets tqdm -q

In [2]:
import json, os, re, time, random
import importlib.metadata
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

from tqdm.auto import tqdm
from datasets import Dataset

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import CrossEncoder
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

print("All imports successful.")

c:\Users\hbahmanyar\MentorApp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


In [3]:
# ── Use OS-level certificate store for SSL ──────────────────────────────────
import truststore
truststore.inject_into_ssl()
print("✓ Injected OS certificate store (truststore) for SSL verification.")

✓ Injected OS certificate store (truststore) for SSL verification.


## 2 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH       = Path("../Datasets/final_dataset.json").resolve()
CHROMA_DIR      = str(Path("../VectorDB/chroma_library_docs").resolve())
OUT_DIR         = Path("RAG_outputs/RAG_with_Baseline").resolve()
COLLECTION_NAME = "library_docs"

# ── Smoke report (source of failed samples + tracebacks) ─────────────────────
SMOKE_REPORT_PATH = Path(
    r"../Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_runtime_A/smoke_report.json"
).resolve()

# ── OpenRouter / LLM ─────────────────────────────────────────────────────────
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
MODEL_ID           = "qwen/qwen2.5-coder-7b-instruct"
MAX_TOKENS         = 8192

# ── Dataset split (to get ground truth) ──────────────────────────────────────
SEED      = 42
TEST_SIZE = 0.15  # → 88 eval samples

# ── RAG settings ─────────────────────────────────────────────────────────────
N_RETRIEVE          = 10     # candidates per library from bi-encoder
N_RERANK            = 3      # final top-k after cross-encoder reranking
MAX_QUERY           = 500    # max chars of error text used as RAG query
MAX_CTX_CHARS       = 2000   # max total chars of RAG context sent to the LLM
MIN_RERANKER_SCORE  = 0.0    # only include docs scored above this threshold

# ── Reranker ─────────────────────────────────────────────────────────────────
RERANKER_MODEL = "BAAI/bge-reranker-base"

print(f"Dataset       : {DATA_PATH}")
print(f"Smoke report  : {SMOKE_REPORT_PATH}")
print(f"ChromaDB      : {CHROMA_DIR}")
print(f"Output dir    : {OUT_DIR}")
print(f"Model         : {MODEL_ID}")
print(f"Retrieve      : {N_RETRIEVE}/lib → rerank → top-{N_RERANK} (score>{MIN_RERANKER_SCORE})")
print(f"Max RAG ctx   : {MAX_CTX_CHARS} chars")

Dataset    : C:\Users\hbahmanyar\MentorApp\Datasets\final_dataset.json
ChromaDB   : C:\Users\hbahmanyar\MentorApp\VectorDB\chroma_library_docs
Output dir : C:\Users\hbahmanyar\MentorApp\RAG_Pipelines\RAG_outputs\RAG_with_Baseline
Model      : qwen/qwen2.5-coder-7b-instruct
MAX_TOKENS : 8192
Reranker   : BAAI/bge-reranker-base
Retrieve   : 10/lib → rerank → top-1 (score>0.0)
Max RAG ctx: 1500 chars


## 3 — Load Smoke Report & Dataset

Load `smoke_report.json` → filter `ok=false` entries with real tracebacks (`returncode=1`).  
Also load the dataset to get ground-truth `correct_code` for each sample.

In [ ]:
random.seed(SEED)

# ── Load ground-truth dataset + eval split ────────────────────────────────────
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
split = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
eval_dataset = split["test"]
eval_examples = [eval_dataset[i] for i in range(len(eval_dataset))]

print(f"Total dataset : {len(data)}")
print(f"Eval split    : {len(eval_examples)}")

# ── Load smoke_report and filter failed entries ───────────────────────────────
with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as f:
    smoke_data = json.load(f)

# Keep only entries with real runtime errors (returncode=1, have tracebacks)
failed_entries = [
    e for e in smoke_data
    if not e["ok"] and e.get("returncode") == 1 and "Traceback" in e.get("stderr_tail", "")
]

print(f"\nSmoke report  : {len(smoke_data)} total entries")
print(f"Failed (ok=false): {sum(1 for e in smoke_data if not e['ok'])}")
print(f"With traceback   : {len(failed_entries)}")

# ── For each failed entry, load prediction.py and attach ground truth ─────────
for entry in failed_entries:
    pred_path = Path(entry["file"])
    entry["prediction_code"] = pred_path.read_text(encoding="utf-8") if pred_path.exists() else ""
    eval_idx = entry["idx"] - 1  # smoke idx is 1-based
    entry["eval_sample"] = eval_examples[eval_idx]

print(f"\nFailed entries ready: {len(failed_entries)}")
for e in failed_entries:
    stderr = e["stderr_tail"]
    err_lines = [l.strip() for l in stderr.split("\n") if l.strip()]
    last_err = err_lines[-1] if err_lines else "?"
    print(f"  idx={e['idx']:2d}  {e['eval_sample']['title'][:45]:<45s}  → {last_err[:80]}")

Total samples : 582
Keys          : ['title', 'description', 'difficulty', 'correct_code', 'incorrect_code', 'error_type']
Train         : 494
Eval          : 88


## 4 — Initialize RAG Retriever & Reranker

- **Bi-encoder** — `BAAI/bge-base-en-v1.5` for fast initial retrieval from ChromaDB  
- **Cross-encoder** — `BAAI/bge-reranker-base` for precise reranking of candidates

The embedding function must match the one used during ingestion.

In [6]:
# ── Embedding function (must match ingestion) ────────────────────────────────
embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5",
    device="cpu",
    normalize_embeddings=True,
)

# ── Connect to ChromaDB ──────────────────────────────────────────────────────
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# ── Initialize Reranker (cross-encoder) ──────────────────────────────────────
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)

print(f"Collection      : {COLLECTION_NAME}")
print(f"Documents       : {collection.count()}")
print(f"Embedding model : BAAI/bge-base-en-v1.5")
print(f"Reranker model  : {RERANKER_MODEL}")

# ══════════════════════════════════════════════════════════════════════════════
# VENV VERSION INTROSPECTION + CHROMA VERSION MATCHING
# ══════════════════════════════════════════════════════════════════════════════

# ── Step 1: Get all installed package versions from the venv ──────────────────
_ALL_INSTALLED = {
    d.metadata["Name"].lower(): d.version
    for d in importlib.metadata.distributions()
}

# Libraries we care about (pip distribution names)
_TRACKED_LIBS = [
    "numpy", "pandas", "scipy", "matplotlib", "seaborn", "scikit-learn",
    "torch", "torchvision", "torchaudio", "tensorflow", "opencv-python",
    "scikit-image", "xgboost", "lightgbm", "catboost", "transformers",
    "datasets", "accelerate", "sentencepiece", "langchain", "pillow",
    "pyarrow", "openpyxl", "requests", "httpx", "tqdm",
]

VENV_VERSIONS: dict[str, str] = {
    lib: _ALL_INSTALLED[lib]
    for lib in _TRACKED_LIBS
    if lib in _ALL_INSTALLED
}

print(f"\nInstalled library versions ({len(VENV_VERSIONS)} tracked):")
for lib, ver in sorted(VENV_VERSIONS.items()):
    print(f"  {lib:20s}  {ver}")

# ── Step 2: Get available versions per library in ChromaDB ────────────────────
_all_meta = collection.get(include=["metadatas"])["metadatas"]
_CHROMA_VERSIONS: dict[str, set[str]] = {}
for m in _all_meta:
    _CHROMA_VERSIONS.setdefault(m["library"], set()).add(m["version"])

print(f"\nChromaDB versions per library:")
for lib in sorted(_CHROMA_VERSIONS):
    print(f"  {lib:20s}  {sorted(_CHROMA_VERSIONS[lib])}")


# ── Step 3: Version matching logic ────────────────────────────────────────────

def _parse_version(v: str) -> tuple:
    """Parse version string into comparable tuple: '2.4.2' → (2, 4, 2)."""
    parts = []
    for p in re.split(r"[.\-]", v):
        try:
            parts.append(int(p))
        except ValueError:
            break
    return tuple(parts) if parts else (0,)


def best_chroma_version(library: str, installed_version: str) -> str | None:
    """
    Find the best matching ChromaDB version for an installed library version.

    Priority:
      1. Exact match (e.g. '3.0.0' == '3.0.0')
      2. Major.minor match (e.g. installed '1.8.0' matches chroma '1.8')
      3. Closest version ≤ installed (most relevant docs)
      4. 'latest' if nothing else available
      5. None if library not in ChromaDB at all
    """
    available = _CHROMA_VERSIONS.get(library)
    if not available:
        return None

    inst_parsed = _parse_version(installed_version)

    # Exact match
    if installed_version in available:
        return installed_version

    # Major.minor prefix match (e.g. chroma has "1.8", installed is "1.8.0")
    inst_prefix = ".".join(str(x) for x in inst_parsed[:2])
    for av in available:
        if av == inst_prefix or av.startswith(inst_prefix + "."):
            return av

    # Closest version ≤ installed (excluding "latest" and non-numeric)
    numeric = []
    for av in available:
        if av == "latest" or av.endswith("x"):
            continue
        numeric.append((av, _parse_version(av)))

    # Filter to versions ≤ installed, pick the highest
    candidates = [(av, parsed) for av, parsed in numeric if parsed <= inst_parsed]
    if candidates:
        return max(candidates, key=lambda x: x[1])[0]

    # Fallback: "latest" if available
    if "latest" in available:
        return "latest"

    # Fallback: any version with "x" suffix (e.g. "4.x", "2.x")
    wildcard = [av for av in available if av.endswith("x")]
    if wildcard:
        return wildcard[0]

    # Last resort: closest available version (even if newer)
    if numeric:
        return min(numeric, key=lambda x: abs(sum(a - b for a, b in zip(x[1], inst_parsed))))[0]

    return None


# ── Build the full mapping: library → best chroma version ─────────────────────
VENV_TO_CHROMA: dict[str, str | None] = {}
for lib, ver in VENV_VERSIONS.items():
    VENV_TO_CHROMA[lib] = best_chroma_version(lib, ver)

print(f"\nVersion mapping (venv → ChromaDB):")
for lib in sorted(VENV_TO_CHROMA):
    inst = VENV_VERSIONS[lib]
    chroma = VENV_TO_CHROMA[lib]
    match_status = "✓" if chroma else "✗ (no docs)"
    print(f"  {lib:20s}  {inst:12s} → {str(chroma):12s}  {match_status}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 561.49it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 305.07it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection      : library_docs
Documents       : 647
Embedding model : BAAI/bge-base-en-v1.5
Reranker model  : BAAI/bge-reranker-base

Installed library versions (26 tracked):
  accelerate            1.12.0
  catboost              1.2.8
  datasets              4.5.0
  httpx                 0.28.1
  langchain             1.2.10
  lightgbm              4.6.0
  matplotlib            3.10.8
  numpy                 2.4.2
  opencv-python         4.13.0.92
  openpyxl              3.1.5
  pandas                3.0.0
  pillow                12.1.1
  pyarrow               23.0.1
  requests              2.32.5
  scikit-image          0.26.0
  scikit-learn          1.8.0
  scipy                 1.17.0
  seaborn               0.13.2
  sentencepiece         0.2.1
  tensorflow            2.20.0
  torch                 2.10.0
  torchaudio            2.10.0
  torchvision           0.25.0
  tqdm                  4.67.3
  transformers          5.2.0
  xgboost               3.2.0

ChromaDB versions per li

## 5 — Initialize LLM Client

In [7]:
llm = ChatOpenAI(
    model=MODEL_ID,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=MAX_TOKENS,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "MentorApp-RAG-Eval",
    },
)


def llm_fix(messages, retries=6, min_backoff=1.0, max_backoff=20.0):
    """Call the LLM with retry / exponential backoff. Returns (content, latency_s)."""
    t0 = time.perf_counter()
    last_err = None

    for attempt in range(retries):
        try:
            resp = llm.invoke(messages)
            return resp.content, time.perf_counter() - t0
        except Exception as e:
            last_err = e
            sleep_s = min(max_backoff, min_backoff * (2 ** attempt)) + random.random()
            print(f"  [WARN] attempt {attempt+1}/{retries} failed ({type(e).__name__}): {e}")
            print(f"         retrying in {sleep_s:.1f}s ...")
            time.sleep(sleep_s)

    raise last_err


print(f"LLM ready: {MODEL_ID}")

LLM ready: qwen/qwen2.5-coder-7b-instruct


## 6 — Prompts, RAG Helpers & Parsing

### Error-based RAG retrieval strategy
1. **Extract library names** from `import` statements in the failed code  
2. **Use the traceback/error text as the RAG query** — query ChromaDB per detected library  
3. **Rerank with cross-encoder**, keep top-k docs above score threshold  
4. **Build simple prompt**: failed code + traceback + relevant docs

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = (
    "You fix Python programs.\n"
    "Return EXACTLY this format and nothing else:\n"
    "<correct_code>\n...full corrected python code...\n</correct_code>\n"
    "<error_type>\n...one short line describing the bug type...\n</error_type>"
)


# ══════════════════════════════════════════════════════════════════════════════
# LIBRARY DETECTION
# ══════════════════════════════════════════════════════════════════════════════

_IMPORT_TO_LIB = {
    "sklearn": "scikit-learn", "skimage": "scikit-image",
    "cv2": "opencv-python", "PIL": "pillow", "Pillow": "pillow",
    "np": "numpy", "pd": "pandas", "tf": "tensorflow",
    "plt": "matplotlib", "sns": "seaborn",
    "xgb": "xgboost", "lgb": "lightgbm", "catboost": "catboost",
    "torch": "torch", "torchvision": "torchvision", "torchaudio": "torchaudio",
    "transformers": "transformers", "datasets": "datasets",
    "accelerate": "accelerate", "sentencepiece": "sentencepiece",
    "langchain": "langchain", "scipy": "scipy",
    "pyarrow": "pyarrow", "openpyxl": "openpyxl",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "tensorflow": "tensorflow",
    "requests": "requests", "httpx": "httpx", "tqdm": "tqdm",
    "statsmodels": "statsmodels", "keras": "tensorflow",
}


def detect_libraries(code: str) -> list[str]:
    """Extract library names from import statements in the code."""
    libs = set()
    for m in re.finditer(r"(?:from|import)\s+([\w.]+)", code):
        top_module = m.group(1).split(".")[0]
        lib = _IMPORT_TO_LIB.get(top_module, top_module)
        libs.add(lib)
    return sorted(libs)


def get_installed_versions(libs: list[str]) -> dict[str, str]:
    """Look up installed versions for the detected libraries."""
    return {lib: VENV_VERSIONS[lib] for lib in libs if lib in VENV_VERSIONS}


# ══════════════════════════════════════════════════════════════════════════════
# TRACEBACK EXTRACTION
# ══════════════════════════════════════════════════════════════════════════════

def extract_traceback(stderr_tail: str) -> str:
    """
    Extract only the Traceback portion from stderr_tail.
    Strips warnings and other noise before the traceback.
    """
    lines = stderr_tail.split("\n")
    # Find the last 'Traceback (most recent call last):' line
    tb_start = None
    for i, line in enumerate(lines):
        if "Traceback (most recent call last):" in line:
            tb_start = i
    if tb_start is not None:
        return "\n".join(lines[tb_start:]).strip()
    # Fallback: return last 5 lines
    return "\n".join(lines[-5:]).strip()


def extract_error_line(stderr_tail: str) -> str:
    """Extract just the final error line (e.g. 'ValueError: ...')."""
    lines = [l.strip() for l in stderr_tail.split("\n") if l.strip()]
    # Walk backwards to find the error line
    for line in reversed(lines):
        if re.match(r"^[A-Z]\w*(Error|Exception|Warning):", line):
            return line
    return lines[-1] if lines else ""


# ══════════════════════════════════════════════════════════════════════════════
# VERSION-AWARE RAG RETRIEVAL (query = error text)
# ══════════════════════════════════════════════════════════════════════════════

def _acceptable_versions(library: str) -> list[str]:
    available = _CHROMA_VERSIONS.get(library, set())
    if not available:
        return []
    accept = set()
    best = VENV_TO_CHROMA.get(library)
    if best:
        accept.add(best)
    if "latest" in available:
        accept.add("latest")
    for v in available:
        if v.endswith("x"):
            accept.add(v)
    return sorted(accept)


def _query_library(lib: str, query_text: str, n_results: int) -> list[tuple[str, dict]]:
    """Query ChromaDB for a single library with version-aware filtering."""
    versions = _acceptable_versions(lib)
    if len(versions) == 1:
        where = {"$and": [{"library": lib}, {"version": versions[0]}]}
    elif len(versions) > 1:
        where = {"$and": [{"library": lib}, {"version": {"$in": versions}}]}
    else:
        where = {"library": lib}
    try:
        results = collection.query(
            query_texts=[query_text], n_results=n_results, where=where,
        )
        return list(zip(results["documents"][0], results["metadatas"][0]))
    except Exception:
        try:
            results = collection.query(
                query_texts=[query_text], n_results=n_results, where={"library": lib},
            )
            return list(zip(results["documents"][0], results["metadatas"][0]))
        except Exception:
            return []


def retrieve_rag_context(
    error_text: str,
    code: str,
    n_retrieve: int = N_RETRIEVE,
    n_rerank: int = N_RERANK,
    max_query_len: int = MAX_QUERY,
    max_ctx_chars: int = MAX_CTX_CHARS,
    min_score: float = MIN_RERANKER_SCORE,
) -> str:
    """
    Error-based two-stage retrieval:
      1. Detect libraries from code imports
      2. Use ERROR TEXT as the RAG query (not the code itself)
      3. Query ChromaDB per library (version-filtered)
      4. Rerank with cross-encoder, filter by min_score
      5. Return formatted context or "" if nothing passes
    """
    libs = detect_libraries(code)
    query_text = error_text[:max_query_len]
    known_libs = [lib for lib in libs if lib in _CHROMA_VERSIONS]

    if not known_libs:
        return ""

    # Stage 1: Per-library bi-encoder retrieval using error as query
    all_candidates = []
    seen_docs = set()
    for lib in known_libs:
        lib_results = _query_library(lib, query_text, n_retrieve)
        for doc, meta in lib_results:
            if doc not in seen_docs:
                all_candidates.append((doc, meta))
                seen_docs.add(doc)

    if not all_candidates:
        return ""

    # Stage 2: Cross-encoder reranking
    pairs = [(query_text, doc) for doc, _ in all_candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(scores, all_candidates), key=lambda x: x[0], reverse=True)
    passing = [(s, cand) for s, cand in scored if s > min_score]

    if not passing:
        return ""

    top_k = passing[:n_rerank]

    parts = []
    total_chars = 0
    for score, (doc, meta) in top_k:
        chunk = f"[{meta['library']} v{meta['version']}]\n{doc}"
        if total_chars + len(chunk) > max_ctx_chars and parts:
            break
        parts.append(chunk)
        total_chars += len(chunk)

    return "\n---\n".join(parts)


# ══════════════════════════════════════════════════════════════════════════════
# USER PROMPT BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_user_prompt(
    failed_code: str,
    traceback_text: str,
    rag_context: str,
) -> str:
    """
    Simple prompt: failed code + traceback + relevant docs.
    Keep it minimal to avoid confusing the model.
    """
    prompt = (
        "Fix this Python code based on the runtime error.\n\n"
        "Traceback:\n"
        f"{traceback_text}\n\n"
        "Code:\n"
        f"{failed_code}"
    )

    if rag_context.strip():
        prompt += f"\n\nReference docs:\n{rag_context}"

    return prompt


# ══════════════════════════════════════════════════════════════════════════════
# RESPONSE PARSING
# ══════════════════════════════════════════════════════════════════════════════

def extract_tag(text: str, tag: str) -> str:
    text = text or ""
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""


def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)
    return s.strip()


def extract_correct_code_and_error(raw: str):
    raw = (raw or "").strip()
    code = extract_tag(raw, "correct_code")
    if not code.strip():
        m = re.search(r"```python\s*(.*?)```", raw, flags=re.DOTALL | re.IGNORECASE)
        if m:
            code = m.group(1)
    if not code.strip():
        m = re.search(r"```\s*(.*?)```", raw, flags=re.DOTALL)
        if m:
            code = m.group(1)
    code = strip_code_fences(code or "")
    err = extract_tag(raw, "error_type").strip()
    return code, err


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def similarity_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def safe_sample_id(idx: int, title: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_]", "_", title)[:40]
    return f"{idx:03d}_{safe}"


def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


print("Prompts, error-based RAG helpers, and parsing functions ready.")

Prompts, score-gated RAG helpers, and parsing functions ready.


## 7 — RAG Retrieval Test (Error-Based)

Quick sanity check — verify that the error-based RAG query retrieves relevant docs for a failed sample.

In [ ]:
# Test on the first failed entry
entry = failed_entries[0]
test_code = entry["prediction_code"]
stderr = entry["stderr_tail"]

tb_text = extract_traceback(stderr)
err_line = extract_error_line(stderr)
libs = detect_libraries(test_code)

print(f"Sample    : {entry['eval_sample']['title']}")
print(f"idx       : {entry['idx']}")
print(f"Error line: {err_line}")
print(f"Libs      : {libs}")
print(f"\n── Traceback ──\n{tb_text[:500]}")

print(f"\n── RAG retrieval (query = error text) ──")
rag_ctx = retrieve_rag_context(tb_text, test_code)
print(f"RAG context length: {len(rag_ctx)} chars")
if rag_ctx:
    print(rag_ctx[:600])
    if len(rag_ctx) > 600:
        print("\n  [... truncated ...]")

print(f"\n\n── User prompt preview ──\n")
user_prompt = build_user_prompt(test_code, tb_text, rag_ctx)
print(user_prompt[:1200])
if len(user_prompt) > 1200:
    print("\n  [... truncated ...]")

Sample : Titanic Missing Age Imputation using SimpleImputer
Detected libs : ['LogisticRegression', 'SimpleImputer', 'accuracy_score', 'dropping', 'numpy', 'pandas', 'scikit-learn', 'seaborn', 'train_test_split', 'warnings']
Installed vers : {'numpy': '2.4.2', 'pandas': '3.0.0', 'scikit-learn': '1.8.0', 'seaborn': '0.13.2'}

  LogisticRegression    installed=N/A           chroma=[]  → filter=[]
  SimpleImputer         installed=N/A           chroma=[]  → filter=[]
  accuracy_score        installed=N/A           chroma=[]  → filter=[]
  dropping              installed=N/A           chroma=[]  → filter=[]
  numpy                 installed=2.4.2         chroma=['2.0.0', '2.3.0', '2.4.0']  → filter=['2.4.0']
  pandas                installed=3.0.0         chroma=['2.3.0', '3.0.0']  → filter=['3.0.0']
  scikit-learn          installed=1.8.0         chroma=['1.7', '1.8']  → filter=['1.8']
  seaborn               installed=0.13.2        chroma=['0.13.0']  → filter=['0.13.0']
  train_test_split

## 8 — Run Error-Based RAG Pipeline

For each failed smoke_report entry:
1. Read the failed `prediction.py` + extract traceback from `stderr_tail`
2. Use **traceback as RAG query** → retrieve relevant docs per library
3. Build prompt: failed code + traceback + RAG docs
4. Call Qwen 2.5 Coder via OpenRouter
5. Parse response → save all artifacts to `RAG_outputs/RAG_with_Baseline/`

In [ ]:
ensure_dir(OUT_DIR)
summary = []
N_SAMPLES = len(failed_entries)

for i, entry in enumerate(tqdm(failed_entries[:N_SAMPLES], desc="RAG Pipeline (error-based)")):
    eval_sample = entry["eval_sample"]
    smoke_idx = entry["idx"]
    title = eval_sample.get("title", "untitled")
    sid = safe_sample_id(smoke_idx, title)
    sample_dir = ensure_dir(OUT_DIR / sid)

    try:
        failed_code = entry["prediction_code"]
        correct = eval_sample.get("correct_code", "")
        err_type = eval_sample.get("error_type", "")
        stderr = entry["stderr_tail"]

        # ── Step 1: Extract traceback + error line ────────────────────────
        tb_text = extract_traceback(stderr)
        err_line = extract_error_line(stderr)

        # ── Step 2: Detect libraries from the failed code ─────────────────
        libs = detect_libraries(failed_code)

        # ── Step 3: RAG retrieval using error text as query ───────────────
        rag_context = retrieve_rag_context(tb_text, failed_code)
        rag_used = bool(rag_context.strip())

        # ── Step 4: Build prompt ──────────────────────────────────────────
        user_prompt = build_user_prompt(failed_code, tb_text, rag_context)
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ]

        # ── Step 5: Call LLM ──────────────────────────────────────────────
        raw_output, latency = llm_fix(messages)

        # ── Step 6: Parse response ────────────────────────────────────────
        if not (raw_output or "").strip():
            pred_code, pred_err, status = "", "", "EMPTY_MODEL_OUTPUT"
        else:
            pred_code, pred_err = extract_correct_code_and_error(raw_output)
            status = "OK" if (pred_code or "").strip() else "UNPARSEABLE_OUTPUT"

        # ── Step 7: Save artifacts ────────────────────────────────────────
        (sample_dir / "prompt.txt").write_text(user_prompt, encoding="utf-8")
        (sample_dir / "rag_context.txt").write_text(rag_context, encoding="utf-8")
        (sample_dir / "raw_model_output.txt").write_text(raw_output or "", encoding="utf-8")
        (sample_dir / "prediction.py").write_text(pred_code or "", encoding="utf-8")
        (sample_dir / "predicted_error_type.txt").write_text(pred_err or "", encoding="utf-8")
        (sample_dir / "failed_code.py").write_text(failed_code or "", encoding="utf-8")
        (sample_dir / "traceback.txt").write_text(tb_text, encoding="utf-8")
        (sample_dir / "stderr_tail.txt").write_text(stderr, encoding="utf-8")
        if correct:
            (sample_dir / "correct.py").write_text(correct, encoding="utf-8")

        # ── Build summary row ─────────────────────────────────────────────
        row = {
            "sid": sid,
            "smoke_idx": smoke_idx,
            "title": title,
            "error_type": err_type,
            "runtime_error": err_line,
            "detected_libs": libs,
            "status": status,
            "latency_s": round(float(latency), 3),
            "pred_len": len(pred_code or ""),
            "failed_code_len": len(failed_code or ""),
            "sim_to_ref": (
                round(similarity_ratio(pred_code, correct), 4)
                if correct and (pred_code or "").strip() else None
            ),
            "predicted_error_type": pred_err or None,
            "rag_context_len": len(rag_context),
            "rag_used": rag_used,
        }
        summary.append(row)
        print(f"  [{i+1}/{N_SAMPLES}] idx={smoke_idx} status={status} rag={rag_used} latency={latency:.1f}s | {err_line[:60]}")

    except Exception as e:
        (sample_dir / "GEN_EXCEPTION.txt").write_text(repr(e), encoding="utf-8")
        summary.append({
            "sid": sid,
            "smoke_idx": smoke_idx,
            "title": title,
            "error_type": eval_sample.get("error_type", ""),
            "runtime_error": "",
            "detected_libs": [],
            "status": "GEN_EXCEPTION",
            "latency_s": None,
            "pred_len": None,
            "failed_code_len": len(entry.get("prediction_code", "") or ""),
            "sim_to_ref": None,
            "predicted_error_type": None,
            "rag_context_len": None,
            "rag_used": None,
            "gen_exception": repr(e),
        })
        print(f"  [{i+1}/{N_SAMPLES}] idx={smoke_idx} EXCEPTION: {e}")
        continue

# ── Save summary ──────────────────────────────────────────────────────────────
summary_path = OUT_DIR / "summary_rag_baseline.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\n✓ Pipeline complete. {len(summary)} samples processed.")
print(f"  Outputs saved to: {OUT_DIR}")
print(f"  Summary saved to: {summary_path.name}")

RAG Pipeline: 100%|██████████| 5/5 [02:30<00:00, 30.03s/it]


✓ Pipeline complete. 5 samples processed.
  Outputs saved to: C:\Users\hbahmanyar\MentorApp\RAG_Pipelines\RAG_outputs\RAG_with_Baseline
  Summary saved to: summary_rag_baseline.json


## 9 — Results Summary

Aggregate statistics from the error-based RAG pipeline run.

In [ ]:
from collections import Counter

status_counts = Counter(row["status"] for row in summary)
latencies     = [r["latency_s"] for r in summary if r["latency_s"] is not None]
sims          = [r["sim_to_ref"] for r in summary if r["sim_to_ref"] is not None]
ctx_lens      = [r["rag_context_len"] for r in summary if r["rag_context_len"] is not None]
rag_used_cnt  = sum(1 for r in summary if r.get("rag_used"))
rag_total     = sum(1 for r in summary if r.get("rag_used") is not None)

print("=" * 70)
print(f"Error-Based RAG Pipeline Summary  ({len(summary)} failed samples)")
print("=" * 70)

print(f"\nStatus:")
for st, cnt in status_counts.most_common():
    print(f"  {st:25s}: {cnt:4d}  ({cnt/len(summary)*100:.1f}%)")

print(f"\nRAG usage: {rag_used_cnt}/{rag_total} samples received RAG context")

if latencies:
    print(f"\nLatency (s):")
    print(f"  Mean={sum(latencies)/len(latencies):.2f}  "
          f"Min={min(latencies):.2f}  Max={max(latencies):.2f}")

if sims:
    print(f"\nSimilarity to reference:")
    print(f"  Mean={sum(sims)/len(sims):.4f}  "
          f"Min={min(sims):.4f}  Max={max(sims):.4f}")

# Per-sample detail
print(f"\nPer-sample detail:")
print(f"{'idx':>4}  {'Title':<40}  {'Status':<10}  {'Sim':>6}  {'RAG?':>5}  {'Runtime Error':<50}")
print("-" * 120)
for r in summary:
    sim_str = f"{r['sim_to_ref']:.3f}" if r['sim_to_ref'] is not None else "  N/A"
    rag_str = "yes" if r.get("rag_used") else "no"
    err_str = (r.get("runtime_error") or "")[:50]
    print(f"{r['smoke_idx']:4d}  {r['title'][:40]:<40}  {r['status']:<10}  {sim_str:>6}  {rag_str:>5}  {err_str:<50}")

RAG Pipeline Summary  (5 samples)

Status:
  OK                       :    3  (60.0%)
  UNPARSEABLE_OUTPUT       :    2  (40.0%)

RAG usage: 5/5 samples received RAG context
           0/5 skipped (no doc scored > 0.0)

Latency (s):
  Mean=8.65  Min=4.20  Max=10.41

Similarity to reference:
  Mean=0.3617  Min=0.0293  Max=1.0000

RAG context length (chars):
  Mean=1451  Min=1325  Max=1485

Per-sample detail:
  #  Title                                     Status         Sim   RAG?  RAG len
--------------------------------------------------------------------------------
  0  Titanic Missing Age Imputation using Sim  UNPARSEABLE_OUTPUT     N/A    yes     1325
  1  CIFAR-10 Deep CNN Data Augmentation       OK           0.029    yes     1484
  2  Digits Misclassification Report           OK           0.056    yes     1484
  3  Titanic Age Imputation Test               OK           1.000    yes     1476
  4  Penguins Domain Adaptation Evaluation     UNPARSEABLE_OUTPUT     N/A    yes     1485
